# Extracción y Almacenamiento (Chunker)
En este notebook vamos a extraer los documentos para el levantamiento de requerimientos y guardarlos en pgvector para usarlos después como contexto.

In [1]:
!source env/bin/activate
%pip install -q pypdf langchain langchain-community langchain-openai psycopg2-binary pgvector tiktoken dotenv

Note: you may need to restart the kernel to use updated packages.


## Configurar variables de entorno

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

db_name = os.getenv("POSTGRES_DB")
db_password = os.getenv("POSTGRES_PASSWORD")
db_user = os.getenv("POSTGRES_USER")
db_host = os.getenv("POSTGRES_HOST", "localhost")
db_port = os.getenv("POSTGRES_PORT", "5432")

# La URL de conexión al contenedor docker que creamos
CONNECTION_STRING = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"


## Cargar documento PDF (Ej. ISO/IEC/IEEE 29148)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

pdf_path = "./docs/iso_29148.pdf"

# Aquí simulamos que cargó el texto para efectos prácticos mientras consigues el documento real.
# Para la carga real descomenta:
# loader = PyPDFLoader(pdf_path)
# documents = loader.load()

documents = [
    Document(page_content="Los requerimientos funcionales especifican los comportamientos que el sistema debe soportar."),
    Document(page_content="Los requerimientos no funcionales definen restricciones o atributos de calidad, como velocidad, seguridad o mantenibilidad."),
    Document(page_content="La norma especifica que un buen requerimiento no debe ser ambiguo, debe estar fundamentado y debe ser verificable o testeable de forma independiente.")
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = text_splitter.split_documents(documents)

print(f"Total de chunks creados: {len(chunks)}")
print("Ejemplo de un chunk:", chunks[0].page_content)

## Guardar en la base de datos (pgvector)

In [ ]:
from langchain_community.vectorstores.pgvector import PGVector
from langchain_openai import OpenAIEmbeddings

# modelo para los embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# crear la colección y guardar embedings en la db
db = PGVector.from_documents(
    documents=chunks,
    embedding=embeddings,
    connection_string=CONNECTION_STRING,
    collection_name="iso_standards",
    pre_delete_collection=True # Limpia e inicializa desde cero si ya existía
)

print("¡Se han guardado todos los chunks exitosamente en pgvector!")